# Türkçe Haber Sınıflandırma

In [ ]:
import os
import re
import sys
import json
import glob
import pickle
import shutil
import time
import warnings
import builtins
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import scipy.sparse as sp

warnings.filterwarnings('ignore')

_builtin_print = builtins.print
def print(*args, **kwargs):
    kwargs.setdefault('flush', True)
    _builtin_print(*args, **kwargs)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count()
USE_DP = N_GPUS > 1

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'GPU sayı : {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i} : {torch.cuda.get_device_name(i)}')

In [ ]:
KEEP_CATEGORIES   = {'spor', 'ekonomi', 'magazin', 'saglik', 'kultur-sanat', 'teknoloji', 'siyaset'}
WORD_MAX_FEATURES = 60_000
CHAR_MAX_FEATURES = 40_000
HIDDEN1           = 1024
HIDDEN2           = 512
DROPOUT           = 0.3
BATCH_SIZE        = 512 if USE_DP else 256
MAX_EPOCHS        = 30
LR                = 1e-3
WEIGHT_DECAY      = 1e-4
PATIENCE          = 5
LABEL_SMOOTH      = 0.05
OVERSAMPLE_MIN    = 3000
N_ENSEMBLE        = 3
OUTPUT_DIR        = '/kaggle/working/model'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Kategoriler  : {sorted(KEEP_CATEGORIES)}')
print(f'Cihaz        : {DEVICE.upper()}')
print(f'DataParallel : {USE_DP}')
print(f'TF-IDF dim   : {WORD_MAX_FEATURES + CHAR_MAX_FEATURES:,}')
print(f'Gizli katman : {HIDDEN1} → {HIDDEN2}')
print(f'Oversample   : min {OVERSAMPLE_MIN}')
print(f'Ensemble     : {N_ENSEMBLE} model')

## 1. Veri Yükleme

In [ ]:
print('Kaggle input:', os.listdir('/kaggle/input'))

candidates = glob.glob('/kaggle/input/**/news', recursive=True)
NEWS_ROOT = None
for c in candidates:
    if os.path.isdir(c):
        NEWS_ROOT = c
        break

if NEWS_ROOT is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 4:
            print(' ' * depth * 2 + root)
    raise FileNotFoundError('news klasörü bulunamadı')

print(f'NEWS_ROOT : {NEWS_ROOT}')

records = []
for category in sorted(os.listdir(NEWS_ROOT)):
    if category not in KEEP_CATEGORIES:
        continue
    cat_dir = os.path.join(NEWS_ROOT, category)
    if not os.path.isdir(cat_dir):
        continue
    for fname in os.listdir(cat_dir):
        if not fname.endswith('.txt'):
            continue
        fpath = os.path.join(cat_dir, fname)
        try:
            with open(fpath, encoding='utf-8', errors='ignore') as f:
                text = f.read().strip()
            if len(text) > 20:
                records.append({'text': text, 'category': category})
        except Exception:
            pass

df = pd.DataFrame(records)
print(f'\nToplam : {len(df):,}')
print(df['category'].value_counts())

## 2. Ön İşleme ve Bölme

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\sğüşıöçĞÜŞİÖÇ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['category'] = df['category'].str.strip().str.lower()
df['clean']    = df['text'].apply(clean_text)
df = df[df['clean'].str.len() > 20].reset_index(drop=True)

labels   = sorted(df['category'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
N_CLASSES = len(labels)
df['label_id'] = df['category'].map(label2id)

print(f'Kategori sayısı: {N_CLASSES}')
for i, l in id2label.items():
    print(f'  {i}  {l}')

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.125, stratify=df['label_id'], random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

X_all  = train_df['clean'].tolist()
y_all  = train_df['label_id'].tolist()
X_test = test_df['clean'].tolist()
y_test = test_df['label_id'].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.125, stratify=y_all, random_state=42
)

print(f'Train : {len(X_train):,}')
print(f'Val   : {len(X_val):,}')
print(f'Test  : {len(X_test):,}')

## 3. TF-IDF + Oversampling

In [ ]:
print('TF-IDF fit ediliyor...')
t0 = time.time()

tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    max_features=WORD_MAX_FEATURES, sublinear_tf=True, min_df=2, max_df=0.95
)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 4),
    max_features=CHAR_MAX_FEATURES, sublinear_tf=True, min_df=3
)

X_train_feat = sp.hstack([
    tfidf_word.fit_transform(X_train),
    tfidf_char.fit_transform(X_train)
], format='csr')
X_val_feat  = sp.hstack([tfidf_word.transform(X_val),  tfidf_char.transform(X_val)],  format='csr')
X_test_feat = sp.hstack([tfidf_word.transform(X_test), tfidf_char.transform(X_test)], format='csr')

TFIDF_DIM = X_train_feat.shape[1]
print(f'TF-IDF dim: {TFIDF_DIM:,}  ({time.time()-t0:.1f}s)')

y_arr  = np.array(y_train)
counts = np.bincount(y_arr)
print('\nOversampling öncesi:')
for cls in range(N_CLASSES):
    print(f'  {id2label[cls]:<16} {counts[cls]:>5}')

X_parts, y_parts = [], []
for cls in range(N_CLASSES):
    mask = y_arr == cls
    Xc, yc = X_train_feat[mask], y_arr[mask]
    n = len(yc)
    if n < OVERSAMPLE_MIN:
        extra = np.random.RandomState(cls).choice(n, OVERSAMPLE_MIN - n, replace=True)
        Xc = sp.vstack([Xc, Xc[extra]])
        yc = np.concatenate([yc, yc[extra]])
    X_parts.append(Xc)
    y_parts.append(yc)

X_train_feat = sp.vstack(X_parts, format='csr')
y_train      = np.concatenate(y_parts).tolist()
perm = np.random.RandomState(42).permutation(len(y_train))
X_train_feat = X_train_feat[perm]
y_train      = [y_train[i] for i in perm]

new_counts = np.bincount(np.array(y_train))
print('\nOversampling sonrası:')
for cls in range(N_CLASSES):
    print(f'  {id2label[cls]:<16} {new_counts[cls]:>5}')
print(f'\nToplam train: {len(y_train):,}')

## 4. Model

In [ ]:
class NewsMLP(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, n_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),  nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden2, n_classes)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def build_model():
    m = NewsMLP(TFIDF_DIM, HIDDEN1, HIDDEN2, N_CLASSES, DROPOUT)
    if USE_DP:
        m = nn.DataParallel(m)
    return m.to(DEVICE)


print(f'Parametre   : {NewsMLP(TFIDF_DIM, HIDDEN1, HIDDEN2, N_CLASSES, DROPOUT).count_parameters():,}')
print(f'DataParallel: {USE_DP}')

## 5. Dataset ve DataLoader

In [ ]:
class TFIDFDataset(Dataset):
    def __init__(self, X_sparse, y_list):
        self.X = X_sparse
        self.y = torch.tensor(y_list, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        row = self.X[idx]
        if sp.issparse(row):
            row = row.toarray().squeeze(0)
        return torch.tensor(row, dtype=torch.float32), self.y[idx]


train_loader = DataLoader(TFIDFDataset(X_train_feat, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(TFIDFDataset(X_val_feat,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(TFIDFDataset(X_test_feat,  y_test),  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')

## 6. Eğitim (Ensemble)

In [ ]:
cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_train)
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE),
    label_smoothing=LABEL_SMOOTH
)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = correct = total = 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for X_b, y_b in loader:
        preds.extend(model(X_b.to(DEVICE)).argmax(1).cpu().tolist())
        labels.extend(y_b.tolist())
    return accuracy_score(labels, preds), f1_score(labels, preds, average='weighted')


ensemble_paths = []

for run in range(N_ENSEMBLE):
    torch.manual_seed(run)
    np.random.seed(run)

    model     = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    ckpt      = os.path.join(OUTPUT_DIR, f'mlp_model_{run}.pt')
    best_f1   = patience_cnt = 0

    print(f'\n--- Model {run+1}/{N_ENSEMBLE} ---')
    print(f"{'Epoch':>5} {'Loss':>8} {'Train%':>7} {'Val%':>6} {'F1':>7} {'s':>4}")
    print('-' * 42)

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        loss, tr_acc    = train_epoch(model, train_loader, optimizer)
        val_acc, val_f1 = evaluate(model, val_loader)
        scheduler.step()

        mark = ''
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_cnt = 0
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state, ckpt)
            mark = ' ✓'
        else:
            patience_cnt += 1

        print(f'{epoch:>5} {loss:>8.4f} {tr_acc:>6.2%} {val_acc:>5.2%} {val_f1:>7.4f} {time.time()-t0:>3.0f}s{mark}')

        if patience_cnt >= PATIENCE:
            print(f'Early stopping @ epoch {epoch}')
            break

    print(f'En iyi Val F1: {best_f1:.4f}')
    ensemble_paths.append(ckpt)

print(f'\nEnsemble tamamlandı: {len(ensemble_paths)} model')

## 7. Test Değerlendirmesi

In [ ]:
@torch.no_grad()
def ensemble_eval(loader):
    models = []
    for p in ensemble_paths:
        m = NewsMLP(TFIDF_DIM, HIDDEN1, HIDDEN2, N_CLASSES, DROPOUT).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models.append(m)

    preds, labels = [], []
    for X_b, y_b in loader:
        X_b = X_b.to(DEVICE)
        logits = torch.stack([m(X_b) for m in models]).mean(0)
        preds.extend(logits.argmax(1).cpu().tolist())
        labels.extend(y_b.tolist())
    return labels, preds


y_true, y_pred = ensemble_eval(test_loader)
test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average='weighted')

print(f'Test Accuracy : {test_acc:.4f}  ({test_acc:.2%})')
print(f'Test F1       : {test_f1:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=[id2label[i] for i in range(N_CLASSES)], digits=4))

## 8. Kaydetme ve ZIP

In [ ]:
for obj, fname in [
    (tfidf_word, 'tfidf_word.pkl'),
    (tfidf_char, 'tfidf_char.pkl'),
]:
    with open(os.path.join(OUTPUT_DIR, fname), 'wb') as f:
        pickle.dump(obj, f)

with open(os.path.join(OUTPUT_DIR, 'mlp_config.json'), 'w') as f:
    json.dump({'input_dim': TFIDF_DIM, 'hidden1': HIDDEN1, 'hidden2': HIDDEN2,
               'n_classes': N_CLASSES, 'dropout': DROPOUT,
               'n_ensemble': N_ENSEMBLE}, f, indent=2)

with open(os.path.join(OUTPUT_DIR, 'id2label.json'), 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, 'label2id.json'), 'w', encoding='utf-8') as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

print('Kaydedilen dosyalar:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024 / 1024
    print(f'  {fname:<28}  {mb:.1f} MB')

zip_path = '/kaggle/working/turkish_news_mlp_model'
shutil.make_archive(zip_path, 'zip', '/kaggle/working', 'model')
print(f'\nZIP: {zip_path}.zip  ({os.path.getsize(zip_path+".zip")/1024/1024:.1f} MB)')
print('Kaggle > Output > turkish_news_mlp_model.zip > Download')

## 9. Hızlı Test

In [ ]:
models_infer = []
for p in ensemble_paths:
    m = NewsMLP(TFIDF_DIM, HIDDEN1, HIDDEN2, N_CLASSES, DROPOUT).to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    models_infer.append(m)


def quick_predict(text):
    clean = clean_text(text)
    feat  = sp.hstack([tfidf_word.transform([clean]), tfidf_char.transform([clean])], format='csr')
    x     = torch.tensor(feat.toarray(), dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        logits = torch.stack([m(x) for m in models_infer]).mean(0)
        probs  = F.softmax(logits, dim=-1)[0]
    pred_id = int(probs.argmax())
    print(f'Metin    : {text[:80]}')
    print(f'Kategori : {id2label[pred_id].upper()}')
    print(f'Güven    : {probs[pred_id]*100:.1f}%')
    print('-' * 55)


ornekler = [
    'Beşiktaş bu akşam Fenerbahçe ile kritik bir derbi maçına çıkıyor.',
    'Merkez Bankası faiz kararını açıkladı, dolar kuru yükseldi.',
    'Yeni iPhone modeli teknoloji dünyasında büyük yankı uyandırdı.',
    'Sağlık Bakanlığı grip aşısı kampanyasını tüm illerde başlattı.',
    'Ünlü oyuncu yeni filmiyle altın küre ödülü kazandı.',
    'Yapay zeka alanında çığır açan yeni bir algoritma geliştirildi.',
    'Muhalefet partisi genel seçimler için adaylarını açıkladı.',
]
q
print('=' * 55)
for t in ornekler:
    quick_predict(t)